# Binance aggTrades → Google Drive
Скачивает **сырые** месячные файлы aggTrades (фьючерсы USD-M) с публичного архива data.binance.vision прямо на твой Google Drive, с проверкой контрольных сумм. Уже скачанные файлы пропускаются, поэтому блокнот можно перезапускать.

1. Runtime → Change runtime type → **CPU** (GPU не нужен).
2. Запусти ячейки по порядку. Первая попросит разрешить доступ к Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
FIRST_MONTH = '2025-10'   # с какого месяца
LAST_MONTH  = '2026-09'   # по какой месяц
OUT = '/content/drive/MyDrive/binance_aggtrades'

In [ ]:
import os, hashlib, urllib.request, time
os.makedirs(OUT, exist_ok=True)
B = 'https://data.binance.vision/data/futures/um/monthly/aggTrades'
def months(a, b):
    y, m = map(int, a.split('-')); Y, M = map(int, b.split('-'))
    while (y, m) <= (Y, M):
        yield f'{y}-{m:02d}'; m += 1
        if m == 13: y, m = y + 1, 1
for mo in months(FIRST_MONTH, LAST_MONTH):
    for s in SYMBOLS:
        name = f'{s}-aggTrades-{mo}.zip'; dst = f'{OUT}/{name}'
        if os.path.exists(dst):
            print('уже есть', name); continue
        t = time.time(); tmp = f'/content/{name}'
        try:
            urllib.request.urlretrieve(f'{B}/{s}/{name}', tmp)
            want = urllib.request.urlopen(f'{B}/{s}/{name}.CHECKSUM').read().decode().split()[0]
        except Exception as e:
            print('нет файла', name, e); continue
        h = hashlib.sha256()
        with open(tmp, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 24), b''): h.update(chunk)
        if h.hexdigest() != want:
            print('ОШИБКА контрольной суммы', name); os.remove(tmp); continue
        os.replace(tmp, dst) if False else __import__('shutil').move(tmp, dst)
        print(f'ok {name} {os.path.getsize(dst)/1e6:.0f} MB за {time.time()-t:.0f} c')
print('готово')

In [ ]:
!ls -la /content/drive/MyDrive/binance_aggtrades | head -40; du -sh /content/drive/MyDrive/binance_aggtrades